# HubSpot Connector Sample

Load HubSpot CRM contacts, companies, deals and their associations into Delta tables: a full load on the first run, then incremental loads. Read-only against HubSpot. Replace all `<PLACEHOLDER>` values before running.

## Prerequisites

1. Upload `connectors/hubspot/hubspot_client.py` to a workspace folder; it is the only helper file. Set `HELPER_DIR` below to that folder.
2. Create a HubSpot service key, or a legacy private app token, with the scopes `crm.objects.contacts.read`, `crm.objects.companies.read` and `crm.objects.deals.read`.
3. In the AIDP Credential Store, create a **Secret Token** credential with one key holding that token. Set `CREDENTIAL_NAME` and `CREDENTIAL_KEY` below. Do not type the token into a cell.
4. The cluster needs the `requests` package and outbound HTTPS access to `api.hubapi.com`. The target catalog must already exist; the schema is created if missing.
5. Run one instance at a time: two overlapping runs would write the same tables and watermarks.

## Configuration

`MODE` is `incremental` or `full`. An object with no stored watermark is loaded in full on its first incremental run. Associations are always re-read in full and overwrite their table, so run them after the objects they start from, in the same run or later.

In [ ]:
import sys

HELPER_DIR = "<WORKSPACE_PATH_TO_HELPER_FOLDER>"  # contains hubspot_client.py
CREDENTIAL_NAME = "<CREDENTIAL_NAME>"  # Credential Store entry holding the HubSpot token
CREDENTIAL_KEY = "<KEY>"  # key inside that entry
CATALOG = "<CATALOG>"
SCHEMA = "<SCHEMA>"
TABLE_PREFIX = ""  # optional prefix for every table name
OBJECTS = ["contacts", "companies", "deals", "associations"]
MODE = "incremental"  # "incremental" or "full"
OVERLAP_SECONDS = 300  # incremental re-reads this far before the watermark
REQUESTS_PER_SECOND = 4.0  # client-side throttle, HubSpot search allows 5 per second
sys.path.insert(0, HELPER_DIR)

## Run the Sync

`run_sync` keeps a watermark per object in `hubspot_sync_state`, runs each selected object, and raises at the end if any object failed. The other objects are still written.

In [ ]:
import hubspot_client as h

token = aidputils.secrets.get(name=CREDENTIAL_NAME, key=CREDENTIAL_KEY)
counts = h.run_sync(
    spark, token, CATALOG, SCHEMA,
    table_prefix=TABLE_PREFIX, objects=OBJECTS, mode=MODE,
    overlap_seconds=OVERLAP_SECONDS, requests_per_second=REQUESTS_PER_SECOND,
)
print(counts)

## Query the Tables

Read a synced table back with Spark.

In [ ]:
spark.table(f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}contacts").show()
spark.table(f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}hubspot_sync_state").show(truncate=False)

## Limits

- Hard deletes are not visible to the search. Archived records are flagged by id on each incremental run, and a full run rebuilds the tables.
- Merged records disappear from HubSpot without being archived; the surviving record lists them, and they are flagged archived.
- Only a fixed set of columns is typed. Every other property is kept in the `raw_json` column.
- Associations are read for contacts to companies, deals to contacts, and deals to companies, from the ids in the synced tables.
- Run one instance at a time. Overlapping runs would write the same tables.